# M2 — Anti-Curriculum: Decision-Only First
**Phase 1** (40%): `L_dec_only` — build latent reasoning

**Phase 2** (40%): blend → `L_cw_wsft` — add explicit supervision

**Phase 3** (20%): `L_cw_wsft + L_dec_ent` — add safety calibration

In [1]:
# Cell 1: Imports and setup
import os, sys, json, math, re
from typing import List, Dict, Any, Tuple

import torch
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer, TrainerCallback

sys.path.insert(0, os.path.expanduser("~/kd_project"))
from shared_utils import (
    RUNS_DIR, MAX_SEQ_LEN, EPOCHS, LR, BATCH_SIZE, GRAD_ACC, SEED,
    W_FORMAT, W_DECISION, W_EXPL, LAMBDA, STUDENTS,
    load_data, load_student,
    get_section_spans, in_any_span,
    compute_alpha, compute_mean_confidence,
    find_decision_span_chars, teacher_section_entropy_mean,
    build_prompt_text, tokenize_and_mask, FlexibleCollator,
)

OUT_DIR = os.path.join(RUNS_DIR, "m2_anti_curriculum")
os.makedirs(OUT_DIR, exist_ok=True)

PHASE1_END = 0.40
PHASE2_END = 0.80

prompts, teacher_rows = load_data()
MEAN_C = compute_mean_confidence(teacher_rows)
print(f"Phases: dec-only [0,{PHASE1_END}], blend [{PHASE1_END},{PHASE2_END}], full [{PHASE2_END},1.0]")

c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Loaded 5000 teacher samples from: data\clinical_pharm_teacher_train_6000.jsonl
Phases: dec-only [0,0.4], blend [0.4,0.8], full [0.8,1.0]


In [2]:
# Cell 2: Dataset (same structure as M1)
class M2Dataset(Dataset):
    def __init__(self, rows, prompts, tokenizer, is_instruct, mean_c):
        self.rows, self.prompts, self.tokenizer = rows, prompts, tokenizer
        self.is_instruct, self.mean_c = is_instruct, mean_c

    def __len__(self): return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        prompt = self.prompts[r["id"]]
        answer = r["teacher_text"]
        prompt_text = build_prompt_text(self.tokenizer, prompt, self.is_instruct)
        input_ids, offsets, labels, answer_start = tokenize_and_mask(self.tokenizer, prompt_text, answer)

        wsft_weights = [0.0] * len(input_ids)
        spans = get_section_spans(answer)
        dec_spans = [(answer_start + s, answer_start + e) for s, e in spans["decision"]]
        expl_spans = [(answer_start + s, answer_start + e) for s, e in spans["explanation"]]
        for i, (s, e) in enumerate(offsets):
            if e <= s: continue
            if s >= answer_start:
                w = W_FORMAT
                if in_any_span(s, e, dec_spans): w = W_DECISION
                if in_any_span(s, e, expl_spans): w = W_EXPL
                wsft_weights[i] = float(w)
        active_w = [w for w in wsft_weights if w > 0]
        if active_w:
            mean_w = sum(active_w) / len(active_w)
            if mean_w > 1e-6: wsft_weights = [w / mean_w if w > 0 else 0.0 for w in wsft_weights]

        alpha = compute_alpha(r, self.mean_c)
        dec_mask = torch.zeros(len(input_ids), dtype=torch.bool)
        dec_span_chars = find_decision_span_chars(answer)
        if dec_span_chars:
            ds, de = dec_span_chars
            full_ds, full_de = answer_start + ds, answer_start + de
            for i, (s, e) in enumerate(offsets):
                if labels[i] != -100 and not (e <= full_ds or s >= full_de):
                    dec_mask[i] = True
        ent_teacher = teacher_section_entropy_mean(r, dec_span_chars)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "wsft_weights": torch.tensor(wsft_weights, dtype=torch.float),
            "alpha": torch.tensor(alpha, dtype=torch.float),
            "dec_mask": dec_mask,
            "ent_teacher": ent_teacher.float(),
        }

print("✅ Dataset ready")

✅ Dataset ready


In [3]:
# Cell 3: Phase-aware Trainer + Logger
class PhaseLogger(TrainerCallback):
    def __init__(self): self.last_phase = None
    def on_step_begin(self, args, state, control, **kwargs):
        if state.max_steps <= 0: return
        p = state.global_step / state.max_steps
        phase = "Phase1(dec-only)" if p < PHASE1_END else "Phase2(blend)" if p < PHASE2_END else "Phase3(full)"
        if phase != self.last_phase:
            print(f"\n🔄 Step {state.global_step}/{state.max_steps} ({p:.0%}): {phase}")
            self.last_phase = phase

class M2Trainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        wsft_weights = inputs.pop("wsft_weights")
        alpha = inputs.pop("alpha")
        dec_mask = inputs.pop("dec_mask")
        ent_teacher = inputs.pop("ent_teacher")

        outputs = model(**inputs)
        logits = outputs.logits
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        shift_wsft_w = wsft_weights[:, 1:].contiguous()
        shift_dec_mask = dec_mask[:, 1:]
        vocab, B = shift_logits.size(-1), shift_logits.size(0)

        token_loss = torch.nn.CrossEntropyLoss(reduction="none")(
            shift_logits.view(-1, vocab), shift_labels.view(-1)).view(shift_labels.size())
        active = (shift_labels != -100).float()

        progress = self.state.global_step / max(1, self.state.max_steps)

        if progress < PHASE1_END:
            # Phase 1: decision-only
            dec_active = shift_dec_mask.float() * active
            loss = (token_loss * dec_active).sum() / dec_active.sum().clamp(min=1.0)

        elif progress < PHASE2_END:
            # Phase 2: linear blend dec_only → cw_wsft
            blend_t = (progress - PHASE1_END) / (PHASE2_END - PHASE1_END)
            dec_active = shift_dec_mask.float() * active
            L_dec = (token_loss * dec_active).sum() / dec_active.sum().clamp(min=1.0)
            w = shift_wsft_w * active
            denom = active.sum(dim=1).clamp(min=1.0)
            L_cw = ((token_loss * w).sum(dim=1) / denom * alpha).mean()
            loss = (1.0 - blend_t) * L_dec + blend_t * L_cw

        else:
            # Phase 3: cw_wsft + entropy matching
            w = shift_wsft_w * active
            denom = active.sum(dim=1).clamp(min=1.0)
            L_cw = ((token_loss * w).sum(dim=1) / denom * alpha).mean()
            probs = torch.softmax(shift_logits, dim=-1)
            ent_tok = -(probs * torch.log(probs + 1e-9)).sum(-1)
            ent_student = torch.zeros(B, device=logits.device)
            for b in range(B):
                m = shift_dec_mask[b]
                if m.any(): ent_student[b] = ent_tok[b][m].mean()
            L_ent = LAMBDA * ((ent_student - ent_teacher.to(logits.device)) ** 2).mean()
            loss = L_cw + L_ent

        return (loss, outputs) if return_outputs else loss

print("✅ Trainer ready")

✅ Trainer ready


In [4]:
# Cell 4: Train all students
for model_name, cfg in STUDENTS.items():
    print(f"\n{'='*50} M2: {model_name} {'='*50}")
    out_path = os.path.join(OUT_DIR, model_name)
    os.makedirs(out_path, exist_ok=True)

    tokenizer, model = load_student(model_name, cfg)
    dataset = M2Dataset(teacher_rows, prompts, tokenizer, cfg["is_instruct"], MEAN_C)
    collator = FlexibleCollator(tokenizer, extra_1d_fields=["wsft_weights", "dec_mask"],
                                extra_scalar_fields=["alpha", "ent_teacher"])

    trainer = M2Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=out_path, num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACC,
            learning_rate=LR, logging_steps=25, save_strategy="no",
            bf16=True, seed=SEED, report_to="none",
            remove_unused_columns=False, dataloader_num_workers=0,
        ),
        train_dataset=dataset, data_collator=collator, callbacks=[PhaseLogger()],
    )
    trainer.train()
    model.save_pretrained(out_path)
    tokenizer.save_pretrained(out_path)
    print(f"✅ {model_name} → {out_path}")
    del model, tokenizer, trainer; torch.cuda.empty_cache()

print("\n✅ M2 complete.")


================================================== M2: qwen25_1p5b_base ==================================================
  Loading qwen25_1p5b_base from Qwen/Qwen2.5-1.5B...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 467.27it/s]


  ✅ qwen25_1p5b_base: 4,358,144 trainable / 892,974,592 total params

🔄 Step 0/314 (0%): Phase1(dec-only)


Step,Training Loss
25,5.381127
50,2.162236
75,1.607575
100,1.528740
125,1.361450
150,3.433746
175,7.892265
200,11.857948
225,15.418276
250,18.919756



🔄 Step 126/314 (40%): Phase2(blend)

🔄 Step 252/314 (80%): Phase3(full)


✅ qwen25_1p5b_base → runs\m2_anti_curriculum\qwen25_1p5b_base

================================================== M2: qwen25_3b_base ==================================================
  Loading qwen25_3b_base from Qwen/Qwen2.5-3B...


Loading weights: 100%|██████████| 434/434 [00:02<00:00, 207.64it/s]


  ✅ qwen25_3b_base: 7,372,800 trainable / 1,706,045,440 total params

🔄 Step 0/314 (0%): Phase1(dec-only)


Step,Training Loss
25,3.032786
50,1.662001
75,1.482536
100,1.248565


KeyboardInterrupt: 